# 08 – Custom Store

The `CreditStore` ABC defines the storage contract. Implement your own
backend (Redis, DynamoDB, SQLite, etc.) by subclassing and providing
the abstract methods.

Below is a minimal in-memory implementation that shows the contract.

### Implement the ABC

In [ ]:
from ducto.interface.base import CreditStore
from ducto.interface.models import (
    BalanceResult, AddCreditsResult, ReserveResult, DeductionResult,
    RefundResult, TeamDeductionResult, CreateTeamResult, TeamBalanceResult,
    TeamMember, AddTeamMemberResult, AllowanceResult, CapCheckResult,
    PricingConfigResult, SetupResult,
)

class MyCustomStore(CreditStore):
    '''Minimal custom store — dict-backed, no persistence.'''

    def __init__(self):
        self._balances: dict[str, int] = {}
        self._reservations: dict[str, int] = {}

    # -- Required: balance / lifecycle ------------------------------------

    def get_balance(self, user_id: str) -> BalanceResult:
        return BalanceResult(user_id=user_id, balance=self._balances.get(user_id, 0))

    def add_credits(self, user_id: str, amount: int, type: str = "adjustment",
                    metadata=None, expires_at=None) -> AddCreditsResult:
        self._balances[user_id] = self._balances.get(user_id, 0) + amount
        return AddCreditsResult(transaction_id="tx", user_id=user_id,
                                amount=amount, new_balance=self._balances[user_id])

    def reserve_credits(self, user_id: str, amount: int, operation_type: str,
                        metadata=None, min_balance: int = 5) -> ReserveResult:
        bal = self._balances.get(user_id, 0)
        rid = "res_" + user_id[:8]
        self._reservations[rid] = amount
        return ReserveResult(reservation_id=rid, user_id=user_id,
                             amount=amount, balance=bal - amount)

    def deduct_credits(self, user_id: str, reservation_id: str, amount: int,
                       idempotency_key=None, metadata=None) -> DeductionResult:
        amt = self._reservations.pop(reservation_id, amount)
        self._balances[user_id] -= amt
        return DeductionResult(transaction_id="ded", user_id=user_id,
                               amount=-amt,
                               balance_after=self._balances[user_id])

    def refund_credits(self, transaction_id: str, amount: int = None,
                       reason: str = None, metadata=None) -> RefundResult:
        return RefundResult(refund_transaction_id="ref", user_id="",
                            original_transaction_id=transaction_id,
                            amount=amount or 0, new_balance=0,
                            reason=reason or "")

    # -- Pricing ----------------------------------------------------------

    def get_active_pricing(self) -> PricingConfigResult | None:
        return None
    def set_active_pricing(self, config, label=None) -> str:
        return "cfg_1"
    def setup_pricing_config(self, config, name="default") -> PricingConfigResult:
        raise NotImplementedError

    # -- Plans ------------------------------------------------------------

    def get_user_plan(self, user_id: str):
        return None
    def set_user_plan(self, user_id: str, plan_id: str):
        pass
    def check_allowance(self, user_id: str) -> AllowanceResult:
        return AllowanceResult(plan_id="", allowance_remaining=0,
                               period_start="", period_end="")
    def increment_usage_window(self, user_id: str, plan_id: str, amount: int):
        pass

    # -- Caps -------------------------------------------------------------

    def set_spend_cap(self, cap):
        pass
    def check_spend_cap(self, user_id: str, model=None, amount=None) -> CapCheckResult:
        return CapCheckResult()

    # -- Analytics --------------------------------------------------------

    def spend_by_user(self, start, end) -> list:
        return []
    def spend_by_model(self, start, end) -> list:
        return []
    def daily_spend(self, start, end) -> list:
        return []
    def top_users(self, limit, start, end) -> list:
        return []
    def aggregate_stats(self, start, end):
        from ducto.interface.models import AggregateStatsRow
        return AggregateStatsRow()

    # -- Sweep ------------------------------------------------------------

    def sweep_expired_credits(self, dry_run=False):
        from ducto.interface.models import SweepResult
        return SweepResult()

    # -- Teams ------------------------------------------------------------

    def create_team(self, name: str, initial_balance=0) -> CreateTeamResult:
        raise NotImplementedError("Teams not supported")
    def get_team_balance(self, team_id: str) -> TeamBalanceResult:
        raise NotImplementedError
    def add_team_member(self, team_id, user_id, role="member", spend_cap=None):
        raise NotImplementedError
    def get_team_members(self, team_id: str):
        raise NotImplementedError
    def deduct_team(self, team_id, user_id, amount, metadata=None):
        raise NotImplementedError

    def setup(self):
        return SetupResult()

custom_store = MyCustomStore()
print("MyCustomStore implements CreditStore ABC.")

### Use with CreditManager

In [ ]:
import uuid
from ducto.manager import CreditManager

manager = CreditManager(custom_store)

user = str(uuid.uuid4())
manager.add_credits(user, 10_000)
print(f"  Balance: {manager.get_balance(user).balance}")

res = manager.reserve_credits(user, 1_000, operation_type="test")
print(f"  Reserved: {res.amount}, bal={res.balance}")